In [ ]:
# Notebook 2: Labeling & Feature Engineering (200k sample, 10 features)

!pip install -q vaderSentiment textstat tqdm pandas matplotlib seaborn

import pandas as pd
import numpy as np
import re
import matplotlib.pyplot as plt
from tqdm.auto import tqdm
from pathlib import Path
from vaderSentiment.vaderSentiment import SentimentIntensityAnalyzer
import warnings
warnings.filterwarnings('ignore')
tqdm.pandas()

from google.colab import drive
drive.mount('/content/drive')

OUT_DIR = Path('/content/drive/MyDrive/AIE_project/outputs')
CONVERSATIONS_PATH = OUT_DIR / 'conversations.csv'
ML_FEATURES_PATH = OUT_DIR / 'ml_features.csv'
PLOTS_DIR = OUT_DIR / 'plots'
PLOTS_DIR.mkdir(exist_ok=True)

Mounted at /content/drive


In [ ]:
# Load full conversations
df_full = pd.read_csv(CONVERSATIONS_PATH)
print(f'Loaded {len(df_full):,} conversations')

# Sample down to 200k rows for ML (random sample)
ML_SAMPLE_SIZE = 200_000
if len(df_full) > ML_SAMPLE_SIZE:
    df = df_full.sample(n=ML_SAMPLE_SIZE, random_state=42).reset_index(drop=True)
    print(f'Sampled to {len(df):,} rows for ML')
else:
    df = df_full.copy()

Loaded 1,503,227 conversations
Sampled to 200,000 rows for ML


In [ ]:
# Define labeling rule
vader = SentimentIntensityAnalyzer()

URGENCY_KEYWORDS = [
    'refund', 'broken', 'not working', 'doesnt work', "doesn't work",
    'cancel', 'cancelled', 'outage', 'down', 'offline', 'urgent',
    'asap', 'immediately', 'emergency', 'critical', 'help',
    'terrible', 'awful', 'horrible', 'useless', 'worst',
    'unacceptable', 'disgusting', 'furious', 'angry', 'frustrated',
    'disappointed', 'scam', 'fraud', 'lie', 'lied', 'stolen',
    'charge', 'overcharged', 'wrong charge', 'double charged',
    'never received', 'never arrived', 'missing', 'lost'
]

TIME_PRESSURE_KEYWORDS = [
    'still', 'days', 'weeks', 'hours', 'since', 'already',
    'waiting', 'waited', 'no response', 'no reply', 'ignored'
]

def extract_signals(text: str) -> dict:
    if not isinstance(text, str) or len(text) == 0:
        return {k: 0 for k in ['has_urgency_kw', 'has_time_pressure',
                'exclamation_count', 'caps_ratio', 'sentiment_neg',
                'text_length', 'question_marks', 'word_count']}
    text_lower = text.lower()
    letters = [c for c in text if c.isalpha()]
    caps_ratio = sum(1 for c in letters if c.isupper()) / max(len(letters), 1)
    sentiment = vader.polarity_scores(text)
    return {
        'has_urgency_kw':    int(any(kw in text_lower for kw in URGENCY_KEYWORDS)),
        'has_time_pressure': int(any(kw in text_lower for kw in TIME_PRESSURE_KEYWORDS)),
        'exclamation_count': text.count('!'),
        'caps_ratio':        round(caps_ratio, 4),
        'sentiment_neg':     round(sentiment['neg'], 4),
        'sentiment_compound':round(sentiment['compound'], 4),
        'text_length':       len(text),
        'question_marks':    text.count('?'),
        'word_count':        len(text.split())
    }

def label_priority(signals: dict) -> int:
    urgency_score = 0
    urgency_score += signals['has_urgency_kw'] * 2
    urgency_score += signals['has_time_pressure'] * 1
    if signals['exclamation_count'] >= 3:
        urgency_score += 2
    elif signals['exclamation_count'] == 2:
        urgency_score += 1
    if signals['caps_ratio'] > 0.5:
        urgency_score += 2
    elif signals['caps_ratio'] > 0.35:
        urgency_score += 1
    if signals['sentiment_neg'] > 0.7:
        urgency_score += 3
    elif signals['sentiment_neg'] > 0.5:
        urgency_score += 2
    elif signals['sentiment_neg'] > 0.3:
        urgency_score += 1
    if signals['text_length'] > 300:
        urgency_score += 2
    elif signals['text_length'] > 200:
        urgency_score += 1
    if signals['question_marks'] >= 2:
        urgency_score += 1
    if signals['word_count'] < 5 and signals['has_urgency_kw']:
        urgency_score += 1
    return 1 if urgency_score >= 4 else 0


In [ ]:
# Apply labeling
print('Extracting signals...')
signals_list = df['text_clean'].progress_apply(extract_signals)
signals_df = pd.DataFrame(signals_list.tolist())
df['priority'] = signals_df.apply(label_priority, axis=1)

print(f"Normal: {(df['priority']==0).sum():,} ({(df['priority']==0).mean()*100:.1f}%)")
print(f"Urgent: {(df['priority']==1).sum():,} ({(df['priority']==1).mean()*100:.1f}%)")

# Save labeled sample (used by Notebook 4)
df.to_csv(OUT_DIR / 'conversations_labeled.csv', index=False)

Extracting signals...


  0%|          | 0/200000 [00:00<?, ?it/s]

Normal: 194,197 (97.1%)
Urgent: 5,803 (2.9%)


In [ ]:
# Feature engineering – only 10 important features
def engineer_features(text: str) -> dict:
    if not isinstance(text, str):
        text = ''
    text_lower = text.lower()
    words = text.split()
    letters = [c for c in text if c.isalpha()]
    caps_letters = [c for c in letters if c.isupper()]
    sentiment = vader.polarity_scores(text)
    return {
        'has_urgency_kw':      int(any(kw in text_lower for kw in URGENCY_KEYWORDS)),
        'exclamation_count':   text.count('!'),
        'has_time_pressure':   int(any(kw in text_lower for kw in TIME_PRESSURE_KEYWORDS)),
        'char_count':          len(text),
        'sentiment_neg':       sentiment['neg'],
        'question_mark_count': text.count('?'),
        'caps_ratio':          len(caps_letters) / max(len(letters), 1),
        'word_count':          len(words),
        'has_help':            int('help' in text_lower),
        'all_caps_words':      sum(1 for w in words if w.isupper() and len(w) > 1),
    }

print('Engineering features (10 features)...')
feat_list = df['text'].progress_apply(engineer_features)
df_features = pd.DataFrame(feat_list.tolist())
df_features['priority'] = df['priority'].values
df_features['tweet_id'] = df['tweet_id'].values

df_features.to_csv(ML_FEATURES_PATH, index=False)
print(f'ML features saved to {ML_FEATURES_PATH}, shape {df_features.shape}')
print(f'Features: {list(df_features.columns)}')

Engineering features (10 features)...


  0%|          | 0/200000 [00:00<?, ?it/s]

ML features saved to /content/drive/MyDrive/AIE_project/outputs/ml_features.csv, shape (200000, 12)
Features: ['has_urgency_kw', 'exclamation_count', 'has_time_pressure', 'char_count', 'sentiment_neg', 'question_mark_count', 'caps_ratio', 'word_count', 'has_help', 'all_caps_words', 'priority', 'tweet_id']
